In [0]:
CREATE OR REPLACE TABLE ORDERS_SALES.GOLD.CUSTOMERS_DIVERSITY_WITH_TOP_CITY
AS
WITH city_counts
AS
(
SELECT
customer_state,
customer_city,
count(*) AS customers_count
FROM ORDERS_SALES.SILVER.CUSTOMERS_ENRICHED
GROUP BY customer_state, customer_city
),
ranked AS
(
    SELECT
    customer_state,
    customer_city,
    customers_count,
    ROW_NUMBER() OVER (PARTITION BY customer_state ORDER BY customers_count DESC) AS rn
    FROM city_counts
),
state_summary
AS
(
    SELECT
    customer_state,
    COUNT(DISTINCT customer_city) AS distinct_cities,
    SUM(customers_count) AS total_customers,
    ROUND(AVG(customers_count), 2) AS avg_customer_per_city
    FROM city_counts
    GROUP BY customer_state
)
SELECT 
s.customer_state,
s.distinct_cities,
s.avg_customer_per_city,
r.customer_city,
r.customers_count
FROM state_summary s JOIN ranked r
ON s.customer_state = r.customer_state
AND r.rn = 1
ORDER BY s.total_customers DESC;

In [0]:
-- VALIDATE RECORD FROM ORDERS_SALES.GOLD.CUSTOMERS_DIVERSITY_WITH_TOP_CITY
SELECT
*
FROM
ORDERS_SALES.GOLD.CUSTOMERS_DIVERSITY_WITH_TOP_CITY
LIMIT 10;